
# E-CUP 2026 — Fine-tuning bi-encoder: Qwen3-Embedding-0.6B + LoRA + symmetric MLP

Это первый supervised fine-tuning нашей bi-encoder ветки на хорошем item-disjoint split.

## Архитектура

```text
product A ─→ shared Qwen3-Embedding-0.6B ─→ eA ─┐
                                                  ├─→ |eA-eB|
product B ─→ shared Qwen3-Embedding-0.6B ─→ eB ─┤
                                                  ├─→ eA*eB
                                                  ├─→ cosine
                                                  ↓
                                           MLP 2049→512→1
                                                  ↓
                                                logit
```

- Base Qwen weights frozen.
- LoRA добавляется в `q_proj / k_proj / v_proj / o_proj`.
- LoRA и MLP обучаются вместе с первого шага.
- Loss: `BCEWithLogitsLoss` на human labels.
- Train: folds `1–4`; validation: fold `0`.
- На validation считаем **два score одновременно**:
  - `AP_mlp` — итоговый MLP score;
  - `AP_cosine` — cosine уже дообученного encoder.

Так мы увидим отдельно, насколько LoRA улучшила geometry embeddings и сколько добавляет pair-head.

## Почему 1 epoch и без frozen warm-up

Для первого baseline я специально убрал отдельный frozen-head stage: меньше движущихся частей и понятнее эксперимент. Новый head и LoRA учатся совместно. Сначала делаем **ровно 1 human epoch**, смотрим clean CV, и только потом решаем, нужен ли второй.

## Основные настройки

- `MAX_LENGTH=512` — тот же cap, что у zero-shot baseline `0.321941`, чтобы не смешивать эффект LoRA с truncation.
- `LoRA r=16, alpha=32, dropout=0.05`.
- pair batch = 4 на одну T4.
- gradient accumulation = 4.
- 2×T4 → effective batch = 32 pairs.
- FP16 + SDPA.
- MLP `2049 → 512 → 1`.
- LoRA LR = `2e-4`, head LR = `1e-3`.
- cosine LR schedule, 5% warmup.

Перед полным запуском есть обязательный smoke run на 20k train pairs.



## 0. Версии

Мы уже ловили проблемы с `transformers 5.x` в Kaggle. Этот notebook рассчитан на:

```text
transformers == 4.57.6
sentence-transformers == 5.1.2
peft == 0.17.1
```

Сначала проверь версии. Если отличаются — выполни install-cell ниже, затем **Restart Session** и запускай notebook сверху.


In [ ]:
import subprocess

PYTHON = "/usr/bin/python3"

# 1. Удаляем то, что Kaggle подсовывает в несовместимом виде
subprocess.run(
    [
        PYTHON, "-m", "pip", "uninstall", "-y",
        "torchao",
        "huggingface-hub",
        "transformers",
        "sentence-transformers",
        "peft",
    ],
    check=False,
)

# 2. Ставим ровно наш рабочий стек.
# --no-deps принципиален: не даём pip снова обновить huggingface-hub
# или начать перекраивать половину Kaggle environment.
subprocess.run(
    [
        PYTHON, "-m", "pip", "install",
        "--no-deps",
        "--force-reinstall",

        "huggingface-hub==0.36.0",
        "transformers==4.57.6",
        "sentence-transformers==5.1.2",
        "peft==0.17.1",
    ],
    check=True,
)

# 3. Проверяем именно НОВЫМ процессом Python,
# таким же окружением потом пользуется torchrun.
check_code = r"""
import importlib.util

import huggingface_hub
import transformers
import sentence_transformers
import peft

print()
print("=== HF STACK ===")
print("huggingface-hub:      ", huggingface_hub.__version__)
print("transformers:         ", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("peft:                 ", peft.__version__)
print("torchao:              ", importlib.util.find_spec("torchao"))

assert huggingface_hub.__version__ == "0.36.0"
assert transformers.__version__ == "4.57.6"
assert sentence_transformers.__version__ == "5.1.2"
assert peft.__version__ == "0.17.1"
assert importlib.util.find_spec("torchao") is None

print()
print("OK: environment is ready")
"""

subprocess.run(
    [PYTHON, "-c", check_code],
    check=True,
)

In [ ]:
import subprocess

PYTHON = "/usr/bin/python3"

code = r"""
import importlib.util
import importlib.metadata as md

spec = importlib.util.find_spec("torchao")
print("torchao spec:", spec)

try:
    print("torchao metadata version:", md.version("torchao"))
except md.PackageNotFoundError:
    print("torchao metadata: NOT INSTALLED")

import transformers
import sentence_transformers
import peft
import huggingface_hub

print()
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("peft:", peft.__version__)
print("huggingface-hub:", huggingface_hub.__version__)
"""

subprocess.run(
    [PYTHON, "-c", code],
    check=True,
)

In [ ]:
import transformers
import sentence_transformers
import peft
import huggingface_hub
import torch

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("peft:", peft.__version__)
print("huggingface-hub:", huggingface_hub.__version__)

### Выполнять только если версии выше отличаются; после этого Restart Session

## 1. Environment

In [ ]:

from pathlib import Path
import json, os, re, sys, time, subprocess
import numpy as np
import pandas as pd
import torch
from huggingface_hub import snapshot_download

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory/1024**3:.2f} GiB")

assert torch.cuda.device_count() >= 2, "Выбери Kaggle Accelerator: GPU T4 x2"


In [ ]:
!nvidia-smi


## 2. Данные

Нужны:
- `items_human.parquet`
- `human_pairs_5fold_item_disjoint.parquet`

Это наш канонический item-disjoint split.


In [ ]:

def find_file(filename, roots=("/kaggle/input", "/kaggle/working")):
    found = []
    for root in roots:
        root = Path(root)
        if root.exists():
            found.extend(root.rglob(filename))
    found = list(dict.fromkeys(found))
    print(filename + ":")
    for p in found:
        print(" ", p)
    if not found:
        raise FileNotFoundError(f"Не найден {filename}")
    if len(found) > 1:
        print("WARNING: найдено несколько файлов; используется первый")
    return found[0]

ITEMS_PATH = find_file("items_human.parquet")
SPLIT_PATH = find_file("human_pairs_5fold_item_disjoint.parquet")



## 3. Preprocessing один раз в Jupyter

Чтобы два DDP worker не парсили 711k JSON attributes каждый, здесь заранее строим тот же canonical text, что использовали в zero-shot:

```text
Name: ...
Category: ...
Attributes:
key: value
...
```


In [ ]:

PREP_DIR = Path("/kaggle/working/ecup_lora_preprocessed")
PREP_DIR.mkdir(parents=True, exist_ok=True)
ITEMS_TEXT_PATH = PREP_DIR / "items_text.parquet"
PAIRS_INDEXED_PATH = PREP_DIR / "pairs_indexed.parquet"
CATEGORIES_PATH = PREP_DIR / "categories.json"
REBUILD_PREPROCESSING = False


In [ ]:

_ws_re = re.compile(r"\s+")

def clean_scalar(x):
    if x is None:
        return ""
    if isinstance(x, (dict, list, tuple)):
        s = json.dumps(x, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    else:
        s = str(x)
    return _ws_re.sub(" ", s).strip()


def format_attributes(attr_text):
    if attr_text is None:
        return ""
    try:
        obj = json.loads(attr_text)
    except Exception:
        return clean_scalar(attr_text)
    if not isinstance(obj, dict):
        return clean_scalar(obj)
    parts = []
    for key in sorted(obj.keys(), key=lambda x: str(x).lower()):
        value = clean_scalar(obj[key])
        if value:
            parts.append(f"{clean_scalar(key)}: {value}")
    return "\n".join(parts)


def make_item_text(row):
    parts = [
        f"Name: {clean_scalar(row['name'])}",
        f"Category: {clean_scalar(row['category'])}",
    ]
    attrs = format_attributes(row["attributes"])
    if attrs:
        parts.append("Attributes:\n" + attrs)
    return "\n".join(parts)


In [ ]:

if REBUILD_PREPROCESSING or not (ITEMS_TEXT_PATH.exists() and PAIRS_INDEXED_PATH.exists() and CATEGORIES_PATH.exists()):
    print("Building preprocessing...")
    t0 = time.perf_counter()

    items = pd.read_parquet(ITEMS_PATH, columns=["id", "name", "attributes", "category"]).reset_index(drop=True)
    pairs = pd.read_parquet(SPLIT_PATH)
    required = {"id1", "id2", "target", "fold"}
    assert required.issubset(pairs.columns), required - set(pairs.columns)
    assert items["id"].is_unique

    items["text"] = [make_item_text(row) for _, row in items.iterrows()]

    categories = sorted(items["category"].astype(str).unique().tolist())
    cat_to_id = {cat: i for i, cat in enumerate(categories)}
    item_cat_id = items["category"].astype(str).map(cat_to_id).to_numpy(dtype=np.int16)

    id_to_idx = pd.Series(np.arange(len(items), dtype=np.int32), index=items["id"].to_numpy())
    idx1 = pairs["id1"].map(id_to_idx)
    idx2 = pairs["id2"].map(id_to_idx)
    assert idx1.notna().all() and idx2.notna().all()

    pairs_out = pd.DataFrame({
        "idx1": idx1.to_numpy(dtype=np.int32),
        "idx2": idx2.to_numpy(dtype=np.int32),
        "target": pairs["target"].to_numpy(dtype=np.float32),
        "fold": pairs["fold"].to_numpy(dtype=np.int8),
    })
    pairs_out["cat_id"] = item_cat_id[pairs_out["idx1"].to_numpy(dtype=np.int64)]

    cat2 = item_cat_id[pairs_out["idx2"].to_numpy(dtype=np.int64)]
    print("Pairs with different categories:", int(np.sum(pairs_out["cat_id"].to_numpy() != cat2)))

    items[["id", "text", "category"]].to_parquet(ITEMS_TEXT_PATH, index=False)
    pairs_out.to_parquet(PAIRS_INDEXED_PATH, index=False)
    with open(CATEGORIES_PATH, "w", encoding="utf-8") as f:
        json.dump(categories, f, ensure_ascii=False, indent=2)

    print(f"Done in {(time.perf_counter()-t0)/60:.2f} min")
else:
    print("Using existing preprocessing")


In [ ]:

pairs_check = pd.read_parquet(PAIRS_INDEXED_PATH)
print("Pairs:", f"{len(pairs_check):,}")
display(
    pairs_check.groupby("fold").agg(
        pairs=("target", "size"),
        positive_rate=("target", "mean"),
    )
)



## 4. Кешируем Qwen до DDP

Сначала checkpoint целиком оказывается локально. В training script rank 0 и rank 1 затем загружают model **последовательно**, а не одновременно — это специально сделано после наших проблем с materialization weights в Kaggle.


In [ ]:

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
MODEL_DIR = snapshot_download(repo_id=MODEL_NAME)
print("MODEL_DIR:", MODEL_DIR)
for p in sorted(Path(MODEL_DIR).iterdir()):
    if p.is_file():
        print(p.name, f"{p.stat().st_size/1024**2:.1f} MB")


## 5. Создаём DDP training script

In [ ]:
TRAIN_SCRIPT_PATH = Path('/kaggle/working/ecup_train_qwen_lora_pair.py')
TRAIN_SCRIPT_PATH.write_text('\nimport argparse\nimport contextlib\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom peft import LoraConfig, TaskType\nfrom sentence_transformers import SentenceTransformer\nfrom sklearn.metrics import average_precision_score\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader, Dataset, DistributedSampler, Subset\nfrom transformers import get_cosine_schedule_with_warmup\n\n\nclass PairDataset(Dataset):\n    def __init__(self, frame):\n        self.i1 = frame["idx1"].to_numpy(dtype=np.int64)\n        self.i2 = frame["idx2"].to_numpy(dtype=np.int64)\n        self.y = frame["target"].to_numpy(dtype=np.float32)\n        self.cat = frame["cat_id"].to_numpy(dtype=np.int16)\n\n    def __len__(self):\n        return len(self.y)\n\n    def __getitem__(self, idx):\n        return int(self.i1[idx]), int(self.i2[idx]), float(self.y[idx]), int(self.cat[idx])\n\n\nclass SymmetricPairBiEncoder(nn.Module):\n    def __init__(self, encoder, embedding_dim, head_hidden=512, head_dropout=0.10):\n        super().__init__()\n        self.encoder = encoder\n        pair_dim = embedding_dim * 2 + 1\n        self.head = nn.Sequential(\n            nn.Linear(pair_dim, head_hidden),\n            nn.GELU(),\n            nn.Dropout(head_dropout),\n            nn.Linear(head_hidden, 1),\n        )\n\n    def forward(self, tokenized, pair_batch_size):\n        out = self.encoder(tokenized)\n        emb = F.normalize(out["sentence_embedding"].float(), p=2, dim=1)\n\n        e1 = emb[:pair_batch_size]\n        e2 = emb[pair_batch_size:2 * pair_batch_size]\n\n        abs_diff = torch.abs(e1 - e2)\n        prod = e1 * e2\n        cosine = torch.sum(prod, dim=1)\n\n        pair_features = torch.cat([abs_diff, prod, cosine.unsqueeze(1)], dim=1)\n        logits = self.head(pair_features).squeeze(1)\n        return logits, cosine\n\n\ndef parse_args():\n    p = argparse.ArgumentParser()\n    p.add_argument("--model-dir", required=True)\n    p.add_argument("--items-text", required=True)\n    p.add_argument("--pairs-indexed", required=True)\n    p.add_argument("--categories-json", required=True)\n    p.add_argument("--output-dir", required=True)\n\n    p.add_argument("--val-fold", type=int, default=0)\n    p.add_argument("--max-length", type=int, default=512)\n    p.add_argument("--pair-batch-size", type=int, default=4)\n    p.add_argument("--grad-accum", type=int, default=4)\n    p.add_argument("--epochs", type=int, default=1)\n\n    p.add_argument("--lora-r", type=int, default=16)\n    p.add_argument("--lora-alpha", type=int, default=32)\n    p.add_argument("--lora-dropout", type=float, default=0.05)\n    p.add_argument("--lr-lora", type=float, default=2e-4)\n    p.add_argument("--lr-head", type=float, default=1e-3)\n    p.add_argument("--weight-decay", type=float, default=0.01)\n    p.add_argument("--warmup-ratio", type=float, default=0.05)\n    p.add_argument("--head-hidden", type=int, default=512)\n    p.add_argument("--head-dropout", type=float, default=0.10)\n    p.add_argument("--max-grad-norm", type=float, default=1.0)\n\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--log-every", type=int, default=100)\n    p.add_argument("--max-train-pairs", type=int, default=0)\n    p.add_argument("--max-val-pairs", type=int, default=0)\n    p.add_argument("--gradient-checkpointing", action="store_true")\n    return p.parse_args()\n\n\ndef setup_dist():\n    dist.init_process_group(backend="nccl")\n    rank = dist.get_rank()\n    world_size = dist.get_world_size()\n    local_rank = int(os.environ["LOCAL_RANK"])\n    torch.cuda.set_device(local_rank)\n    return rank, world_size, local_rank, torch.device("cuda", local_rank)\n\n\ndef seed_everything(seed, rank):\n    s = seed + rank\n    random.seed(s)\n    np.random.seed(s)\n    torch.manual_seed(s)\n    torch.cuda.manual_seed_all(s)\n\n\ndef maybe_sample(frame, n, seed):\n    if not n or n <= 0 or n >= len(frame):\n        return frame.reset_index(drop=True)\n    return frame.sample(n=n, random_state=seed).reset_index(drop=True)\n\n\ndef load_encoder_sequentially(args, rank, world_size, device):\n    # Sequential loading avoids two Kaggle workers materializing the same checkpoint simultaneously.\n    encoder = None\n    for turn in range(world_size):\n        if rank == turn:\n            print(f"[rank {rank}] loading base model on {device} ...", flush=True)\n            t0 = time.perf_counter()\n            encoder = SentenceTransformer(\n                args.model_dir,\n                device=str(device),\n                local_files_only=True,\n                model_kwargs={\n                    "torch_dtype": torch.float16,\n                    "attn_implementation": "sdpa",\n                },\n                tokenizer_kwargs={"padding_side": "left"},\n            )\n            encoder.max_seq_length = args.max_length\n            print(f"[rank {rank}] base model loaded in {time.perf_counter()-t0:.1f}s", flush=True)\n        dist.barrier()\n\n    lora_cfg = LoraConfig(\n        task_type=TaskType.FEATURE_EXTRACTION,\n        inference_mode=False,\n        r=args.lora_r,\n        lora_alpha=args.lora_alpha,\n        lora_dropout=args.lora_dropout,\n        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],\n        bias="none",\n    )\n    encoder.add_adapter(lora_cfg)\n\n    if args.gradient_checkpointing:\n        auto_model = encoder[0].auto_model\n        if hasattr(auto_model.config, "use_cache"):\n            auto_model.config.use_cache = False\n        if hasattr(auto_model, "enable_input_require_grads"):\n            auto_model.enable_input_require_grads()\n        auto_model.gradient_checkpointing_enable()\n\n    return encoder\n\n\ndef make_text_batch(all_texts, idx1, idx2):\n    a = [all_texts[int(i)] for i in idx1.tolist()]\n    b = [all_texts[int(i)] for i in idx2.tolist()]\n    return a + b\n\n\n@torch.no_grad()\ndef distributed_validate(ddp_model, encoder, all_texts, val_dataset, categories, args,\n                         rank, world_size, device, output_dir, tag):\n    ddp_model.eval()\n    indices = list(range(rank, len(val_dataset), world_size))\n    loader = DataLoader(\n        Subset(val_dataset, indices),\n        batch_size=args.pair_batch_size,\n        shuffle=False,\n        num_workers=0,\n        pin_memory=True,\n    )\n\n    local_logits, local_cosines, local_targets, local_cats = [], [], [], []\n\n    for idx1, idx2, y, cat in loader:\n        pair_bs = len(y)\n        texts = make_text_batch(all_texts, idx1, idx2)\n        tokenized = encoder.tokenize(texts)\n        tokenized = {k: v.to(device, non_blocking=True) for k, v in tokenized.items()}\n\n        with torch.autocast(device_type="cuda", dtype=torch.float16):\n            logits, cosine = ddp_model(tokenized, pair_bs)\n\n        local_logits.append(logits.float().cpu().numpy())\n        local_cosines.append(cosine.float().cpu().numpy())\n        local_targets.append(y.numpy().astype(np.float32))\n        local_cats.append(cat.numpy().astype(np.int16))\n\n    def cat_or_empty(parts, dtype):\n        return np.concatenate(parts) if parts else np.array([], dtype=dtype)\n\n    np.savez_compressed(\n        output_dir / f"_val_{tag}_rank{rank}.npz",\n        logits=cat_or_empty(local_logits, np.float32),\n        cosines=cat_or_empty(local_cosines, np.float32),\n        targets=cat_or_empty(local_targets, np.float32),\n        cats=cat_or_empty(local_cats, np.int16),\n    )\n    dist.barrier()\n\n    if rank == 0:\n        parts = [np.load(output_dir / f"_val_{tag}_rank{r}.npz") for r in range(world_size)]\n        logits = np.concatenate([x["logits"] for x in parts])\n        cosines = np.concatenate([x["cosines"] for x in parts])\n        targets = np.concatenate([x["targets"] for x in parts])\n        cats = np.concatenate([x["cats"] for x in parts])\n\n        rows = []\n        for cat_id, cat_name in enumerate(categories):\n            mask = cats == cat_id\n            if not np.any(mask):\n                continue\n            y = targets[mask]\n            if np.sum(y > 0.5) == 0:\n                ap_mlp = np.nan\n                ap_cos = np.nan\n            else:\n                ap_mlp = float(average_precision_score(y, logits[mask]))\n                ap_cos = float(average_precision_score(y, cosines[mask]))\n            rows.append({\n                "category": cat_name,\n                "pairs": int(mask.sum()),\n                "positives": int(np.sum(y > 0.5)),\n                "positive_rate": float(np.mean(y)),\n                "AP_mlp": ap_mlp,\n                "AP_cosine": ap_cos,\n            })\n\n        per_cat = pd.DataFrame(rows)\n        metrics = {\n            "macro_ap_mlp": float(per_cat["AP_mlp"].mean()),\n            "macro_ap_cosine": float(per_cat["AP_cosine"].mean()),\n            "n_val_pairs": int(len(targets)),\n        }\n        per_cat.to_csv(output_dir / f"per_category_{tag}.csv", index=False)\n        np.savez_compressed(\n            output_dir / f"val_scores_{tag}.npz",\n            logits=logits, cosines=cosines, targets=targets, cats=cats,\n        )\n        with open(output_dir / f"metrics_{tag}.json", "w", encoding="utf-8") as f:\n            json.dump(metrics, f, ensure_ascii=False, indent=2)\n\n        print(\n            f"[VAL {tag}] MLP Macro AP={metrics[\'macro_ap_mlp\']:.6f} | "\n            f"fine-tuned cosine Macro AP={metrics[\'macro_ap_cosine\']:.6f} | "\n            f"pairs={len(targets):,}",\n            flush=True,\n        )\n\n        for r in range(world_size):\n            p = output_dir / f"_val_{tag}_rank{r}.npz"\n            if p.exists():\n                p.unlink()\n\n    dist.barrier()\n    ddp_model.train()\n\n\ndef main():\n    args = parse_args()\n    rank, world_size, local_rank, device = setup_dist()\n    seed_everything(args.seed, rank)\n\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    if rank == 0:\n        print("=" * 100, flush=True)\n        print("E-CUP Qwen3-Embedding-0.6B + LoRA + symmetric MLP", flush=True)\n        print(json.dumps(vars(args), indent=2, ensure_ascii=False), flush=True)\n        print("=" * 100, flush=True)\n\n    items = pd.read_parquet(args.items_text)\n    pairs = pd.read_parquet(args.pairs_indexed)\n    all_texts = items["text"].tolist()\n    with open(args.categories_json, encoding="utf-8") as f:\n        categories = json.load(f)\n\n    train_df = maybe_sample(pairs[pairs["fold"] != args.val_fold].copy(), args.max_train_pairs, args.seed)\n    val_df = maybe_sample(pairs[pairs["fold"] == args.val_fold].copy(), args.max_val_pairs, args.seed + 123)\n    train_dataset = PairDataset(train_df)\n    val_dataset = PairDataset(val_df)\n\n    train_sampler = DistributedSampler(\n        train_dataset,\n        num_replicas=world_size,\n        rank=rank,\n        shuffle=True,\n        seed=args.seed,\n        drop_last=False,\n    )\n    train_loader = DataLoader(\n        train_dataset,\n        batch_size=args.pair_batch_size,\n        sampler=train_sampler,\n        num_workers=0,\n        pin_memory=True,\n        drop_last=False,\n    )\n\n    if rank == 0:\n        print(f"Train pairs: {len(train_dataset):,} | Val pairs: {len(val_dataset):,}", flush=True)\n\n    encoder = load_encoder_sequentially(args, rank, world_size, device)\n    embedding_dim = int(encoder.get_sentence_embedding_dimension())\n    pair_model = SymmetricPairBiEncoder(\n        encoder,\n        embedding_dim,\n        head_hidden=args.head_hidden,\n        head_dropout=args.head_dropout,\n    ).to(device)\n\n    lora_params = [p for p in pair_model.encoder.parameters() if p.requires_grad]\n    head_params = list(pair_model.head.parameters())\n    n_lora = sum(p.numel() for p in lora_params)\n    n_head = sum(p.numel() for p in head_params)\n    n_total = sum(p.numel() for p in pair_model.parameters())\n\n    if rank == 0:\n        print(f"Embedding dim: {embedding_dim}", flush=True)\n        print(f"Trainable LoRA params: {n_lora:,}", flush=True)\n        print(f"Trainable head params: {n_head:,}", flush=True)\n        print(f"Trainable fraction: {(n_lora+n_head)/n_total:.4%}", flush=True)\n\n    ddp_model = DDP(\n        pair_model,\n        device_ids=[local_rank],\n        output_device=local_rank,\n        broadcast_buffers=False,\n        find_unused_parameters=False,\n    )\n\n    optimizer = torch.optim.AdamW(\n        [\n            {"params": lora_params, "lr": args.lr_lora},\n            {"params": head_params, "lr": args.lr_head},\n        ],\n        weight_decay=args.weight_decay,\n    )\n\n    updates_per_epoch = math.ceil(len(train_loader) / args.grad_accum)\n    total_updates = updates_per_epoch * args.epochs\n    warmup_updates = max(1, int(total_updates * args.warmup_ratio))\n    scheduler = get_cosine_schedule_with_warmup(\n        optimizer,\n        num_warmup_steps=warmup_updates,\n        num_training_steps=total_updates,\n    )\n    criterion = nn.BCEWithLogitsLoss()\n    scaler = torch.amp.GradScaler("cuda", enabled=True)\n\n    if rank == 0:\n        effective_batch = args.pair_batch_size * world_size * args.grad_accum\n        print(f"Micro pair batch/GPU: {args.pair_batch_size}", flush=True)\n        print(f"Gradient accumulation: {args.grad_accum}", flush=True)\n        print(f"Effective pair batch: {effective_batch}", flush=True)\n        print(f"Optimizer updates/epoch: {updates_per_epoch:,}", flush=True)\n\n    global_update = 0\n    train_t0 = time.perf_counter()\n    torch.cuda.reset_peak_memory_stats(device)\n\n    for epoch in range(args.epochs):\n        train_sampler.set_epoch(epoch)\n        ddp_model.train()\n        optimizer.zero_grad(set_to_none=True)\n        running_loss = 0.0\n        running_micro = 0\n        epoch_t0 = time.perf_counter()\n\n        for micro_step, (idx1, idx2, y, cat) in enumerate(train_loader):\n            y = y.to(device, dtype=torch.float32, non_blocking=True)\n            pair_bs = len(y)\n            texts = make_text_batch(all_texts, idx1, idx2)\n            tokenized = encoder.tokenize(texts)\n            tokenized = {k: v.to(device, non_blocking=True) for k, v in tokenized.items()}\n\n            last_micro = (micro_step + 1 == len(train_loader))\n            sync_now = (((micro_step + 1) % args.grad_accum == 0) or last_micro)\n            sync_ctx = contextlib.nullcontext() if sync_now else ddp_model.no_sync()\n\n            with sync_ctx:\n                with torch.autocast(device_type="cuda", dtype=torch.float16):\n                    logits, _ = ddp_model(tokenized, pair_bs)\n                    loss_raw = criterion(logits, y)\n                    loss = loss_raw / args.grad_accum\n                scaler.scale(loss).backward()\n\n            running_loss += float(loss_raw.detach().cpu())\n            running_micro += 1\n\n            if sync_now:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(\n                    [p for p in ddp_model.parameters() if p.requires_grad],\n                    args.max_grad_norm,\n                )\n                scaler.step(optimizer)\n                scaler.update()\n                optimizer.zero_grad(set_to_none=True)\n                scheduler.step()\n                global_update += 1\n\n                if rank == 0 and global_update % args.log_every == 0:\n                    elapsed = time.perf_counter() - train_t0\n                    processed_pairs = min(\n                        global_update * args.grad_accum * args.pair_batch_size * world_size,\n                        len(train_dataset) * (epoch + 1),\n                    )\n                    pairs_per_sec = processed_pairs / elapsed if elapsed > 0 else 0.0\n                    avg_loss = running_loss / max(running_micro, 1)\n                    peak_gb = torch.cuda.max_memory_allocated(device) / 1024**3\n                    print(\n                        f"[epoch {epoch+1}/{args.epochs}] update {global_update}/{total_updates} | "\n                        f"loss={avg_loss:.5f} | pairs/s~{pairs_per_sec:.2f} | "\n                        f"lr_lora={optimizer.param_groups[0][\'lr\']:.2e} | "\n                        f"peakVRAM(rank0)={peak_gb:.2f}GB",\n                        flush=True,\n                    )\n                    running_loss = 0.0\n                    running_micro = 0\n\n        dist.barrier()\n        if rank == 0:\n            print(f"Epoch {epoch+1} training wall time: {(time.perf_counter()-epoch_t0)/60:.2f} min", flush=True)\n\n        distributed_validate(\n            ddp_model, encoder, all_texts, val_dataset, categories, args,\n            rank, world_size, device, output_dir, tag=f"epoch{epoch+1}"\n        )\n\n        if rank == 0:\n            raw_model = ddp_model.module\n            ckpt_dir = output_dir / f"checkpoint_epoch{epoch+1}"\n            ckpt_dir.mkdir(parents=True, exist_ok=True)\n            raw_model.encoder.save_pretrained(str(ckpt_dir / "encoder_lora"))\n            torch.save(raw_model.head.state_dict(), ckpt_dir / "pair_head.pt")\n            try:\n                torch.save(raw_model.encoder.get_adapter_state_dict(), ckpt_dir / "adapter_state_dict.pt")\n            except Exception as e:\n                print("Warning: explicit adapter state save failed:", repr(e), flush=True)\n\n            cfg = {\n                **vars(args),\n                "embedding_dim": embedding_dim,\n                "trainable_lora_params": n_lora,\n                "trainable_head_params": n_head,\n                "effective_pair_batch": args.pair_batch_size * world_size * args.grad_accum,\n            }\n            with open(ckpt_dir / "training_config.json", "w", encoding="utf-8") as f:\n                json.dump(cfg, f, ensure_ascii=False, indent=2)\n            print(f"Saved checkpoint: {ckpt_dir}", flush=True)\n\n        dist.barrier()\n\n    if rank == 0:\n        print(f"TRAINING COMPLETE in {(time.perf_counter()-train_t0)/60:.2f} min", flush=True)\n\n    dist.destroy_process_group()\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Saved:', TRAIN_SCRIPT_PATH)


In [ ]:
from pathlib import Path

path = Path("/kaggle/working/ecup_train_qwen_lora_pair.py")

text = path.read_text(encoding="utf-8")

old = """    encoder.add_adapter(lora_cfg)

    if args.gradient_checkpointing:
"""

new = """    encoder.add_adapter(lora_cfg)

    # Base Qwen stays FP16 and frozen.
    # Trainable LoRA parameters must be FP32 for AMP + GradScaler.
    for param in encoder.parameters():
        if param.requires_grad:
            param.data = param.data.float()

    if rank == 0:
        trainable_dtypes = {
            str(p.dtype)
            for p in encoder.parameters()
            if p.requires_grad
        }
        print(
            "Trainable encoder dtypes:",
            trainable_dtypes,
            flush=True,
        )

    if args.gradient_checkpointing:
"""

assert old in text, "Не нашёл место для патча — скрипт отличается от ожидаемого"

text = text.replace(old, new, 1)

path.write_text(text, encoding="utf-8")

print("LoRA FP32 patch applied successfully")


## 6. Конфигурация

`PAIR_BATCH_PER_GPU=4` = 4 пары = 8 текстов на одну T4 за microbatch.

Если smoke даёт OOM:
- `PAIR_BATCH_PER_GPU = 2`
- `GRAD_ACCUM = 8`
- effective batch останется 32
- при необходимости `GRADIENT_CHECKPOINTING = True`


In [ ]:

VAL_FOLD = 0
MAX_LENGTH = 512
PAIR_BATCH_PER_GPU = 4
GRAD_ACCUM = 4
EPOCHS = 1

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LR_LORA = 2e-4
LR_HEAD = 1e-3
HEAD_HIDDEN = 512
HEAD_DROPOUT = 0.10
GRADIENT_CHECKPOINTING = False
SEED = 42

print("Effective pair batch:", PAIR_BATCH_PER_GPU * 2 * GRAD_ACCUM)



## 7. Smoke run — обязательно

20k train pairs + 10k val pairs. Это проверка pipeline, VRAM и скорости, а не финальная метрика.


In [ ]:

# SMOKE_DIR = Path("/kaggle/working/ecup_qwen_lora_smoke")

# cmd = [
#     sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=2",
#     str(TRAIN_SCRIPT_PATH),
#     "--model-dir", str(MODEL_DIR),
#     "--items-text", str(ITEMS_TEXT_PATH),
#     "--pairs-indexed", str(PAIRS_INDEXED_PATH),
#     "--categories-json", str(CATEGORIES_PATH),
#     "--output-dir", str(SMOKE_DIR),
#     "--val-fold", str(VAL_FOLD),
#     "--max-length", str(MAX_LENGTH),
#     "--pair-batch-size", str(PAIR_BATCH_PER_GPU),
#     "--grad-accum", str(GRAD_ACCUM),
#     "--epochs", "1",
#     "--lora-r", str(LORA_R),
#     "--lora-alpha", str(LORA_ALPHA),
#     "--lora-dropout", str(LORA_DROPOUT),
#     "--lr-lora", str(LR_LORA),
#     "--lr-head", str(LR_HEAD),
#     "--head-hidden", str(HEAD_HIDDEN),
#     "--head-dropout", str(HEAD_DROPOUT),
#     "--seed", str(SEED),
#     "--log-every", "50",
#     "--max-train-pairs", "20000",
#     "--max-val-pairs", "10000",
# ]
# if GRADIENT_CHECKPOINTING:
#     cmd.append("--gradient-checkpointing")

# env = os.environ.copy()
# env["HF_HUB_OFFLINE"] = "1"
# env["TOKENIZERS_PARALLELISM"] = "false"

# print("COMMAND:\n", " ".join(cmd))
# t0 = time.perf_counter()
# result = subprocess.run(cmd, env=env)
# smoke_wall = time.perf_counter() - t0
# print(f"Smoke wall time: {smoke_wall/60:.2f} min")
# assert result.returncode == 0, f"Smoke failed: {result.returncode}"



### Что смотреть в smoke log
/
Нужны строки примерно такого вида:

```text
Trainable LoRA params: ...
Trainable head params: ...
Effective pair batch: 32
[epoch 1/1] update ... loss=... pairs/s~... peakVRAM(rank0)=...GB
```

Если peak VRAM уже близко к 15 GB — уменьшаем pair batch до 2.


## 8. Грубая оценка полного времени

In [ ]:

# n_train_full = int((pairs_check["fold"] != VAL_FOLD).sum())
# rough_upper_min = smoke_wall / 60 * n_train_full / 20000
# print("Full train pairs:", f"{n_train_full:,}")
# print("Very rough upper estimate from total smoke wall:", f"{rough_upper_min:.1f} min")
# print("Лучше ориентироваться на pairs/s из train log: smoke включает fixed overhead model loading + validation")


In [ ]:
from pathlib import Path

path = Path("/kaggle/working/ecup_train_qwen_lora_pair.py")
text = path.read_text(encoding="utf-8")

val_block = '''        distributed_validate(
            ddp_model, encoder, all_texts, val_dataset, categories, args,
            rank, world_size, device, output_dir, tag=f"epoch{epoch+1}"
        )
'''

save_start_marker = '''        if rank == 0:
            raw_model = ddp_model.module
'''

# Находим validation внутри training loop
val_start = text.find(val_block)
assert val_start != -1, "Не нашёл блок distributed_validate"

val_end = val_start + len(val_block)

# Сразу после validation сейчас находится сохранение checkpoint
save_start = text.find(save_start_marker, val_end)
assert save_start != -1, "Не нашёл блок сохранения checkpoint"

# После save-блока в текущем скрипте идёт dist.barrier()
barrier_marker = "        dist.barrier()\n"
save_end = text.find(barrier_marker, save_start)
assert save_end != -1, "Не нашёл barrier после save"

save_block = text[save_start:save_end]

# Удаляем старые validation + save + barrier
old_chunk_end = save_end + len(barrier_marker)

# Новый порядок:
# train завершился
# -> SAVE
# -> barrier (ждём пока rank0 закончит запись)
# -> validation
new_chunk = (
    save_block
    + barrier_marker
    + "\n"
    + val_block
)

text = (
    text[:val_start]
    + new_chunk
    + text[old_chunk_end:]
)

path.write_text(text, encoding="utf-8")

print("OK: checkpoint теперь сохраняется ДО validation")


## 9. Полный human fine-tuning

Первый запуск = ровно 1 epoch. После него получаем clean fold0 Macro AP.


In [ ]:

FULL_DIR = Path("/kaggle/working/ecup_qwen_lora_full")

cmd = [
    sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=2",
    str(TRAIN_SCRIPT_PATH),
    "--model-dir", str(MODEL_DIR),
    "--items-text", str(ITEMS_TEXT_PATH),
    "--pairs-indexed", str(PAIRS_INDEXED_PATH),
    "--categories-json", str(CATEGORIES_PATH),
    "--output-dir", str(FULL_DIR),
    "--val-fold", str(VAL_FOLD),
    "--max-length", str(MAX_LENGTH),
    "--pair-batch-size", str(PAIR_BATCH_PER_GPU),
    "--grad-accum", str(GRAD_ACCUM),
    "--epochs", str(EPOCHS),
    "--lora-r", str(LORA_R),
    "--lora-alpha", str(LORA_ALPHA),
    "--lora-dropout", str(LORA_DROPOUT),
    "--lr-lora", str(LR_LORA),
    "--lr-head", str(LR_HEAD),
    "--head-hidden", str(HEAD_HIDDEN),
    "--head-dropout", str(HEAD_DROPOUT),
    "--seed", str(SEED),
    "--log-every", "100",
]
if GRADIENT_CHECKPOINTING:
    cmd.append("--gradient-checkpointing")

env = os.environ.copy()
env["HF_HUB_OFFLINE"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"

print("COMMAND:\n", " ".join(cmd))
t0 = time.perf_counter()
result = subprocess.run(cmd, env=env)
full_wall = time.perf_counter() - t0
print(f"FULL wall time: {full_wall/60:.2f} min")
assert result.returncode == 0, f"Full training failed: {result.returncode}"


## 10. Результаты

In [ ]:

with open(FULL_DIR / "metrics_epoch1.json", encoding="utf-8") as f:
    metrics = json.load(f)
per_category = pd.read_csv(FULL_DIR / "per_category_epoch1.csv")

print("ZERO-SHOT cosine reference @512: 0.321941")
print("Fine-tuned cosine Macro AP:", f"{metrics['macro_ap_cosine']:.6f}")
print("LoRA + MLP Macro AP       :", f"{metrics['macro_ap_mlp']:.6f}")
display(per_category.sort_values("AP_mlp"))



### Как читать два результата

- cosine сильно вырос + MLP ещё добавил → полезны и LoRA geometry, и pair-head.
- cosine почти равен MLP → head почти не нужен.
- cosine почти не вырос, MLP сильно вырос → основную exact-matching работу делает head.


## 11. Сохраняемые artifacts

In [ ]:

for p in sorted(FULL_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(FULL_DIR), f"{p.stat().st_size/1024**2:.1f} MB")



Основные файлы:

```text
ecup_qwen_lora_full/
├── metrics_epoch1.json
├── per_category_epoch1.csv
├── val_scores_epoch1.npz
└── checkpoint_epoch1/
    ├── encoder_lora/
    ├── pair_head.pt
    ├── adapter_state_dict.pt
    └── training_config.json
```

После первого результата следующий порядок я бы делал такой:
1. hard-negative oversampling;
2. второй human epoch только если validation ещё растёт;
3. `r=8/16/32` ablation;
4. LLM pretraining → human final fine-tune;
5. distillation от clean cross-encoder;
6. runtime optimization: 384/256 tokens + приоритетный порядок SKU-critical attributes.
